# Week12 Attribution setup — frozen 150-query diagnostic

VALIDATION ONLY. No confirmation/TEST access. Default Run All uses clearly labelled synthetic fixtures. Real cached setup is allowed in J; real perturbations require explicit RUN_FULL in K. L reads saved results. Evidence attribution means context-dependent input ablation, not internal-model explanation. Optional attention/replacement/third-system studies remain disabled.

Order: H saved analysis → J (`RUN_SMOKE=False`, cached preparation) → K (`RUN_FULL=True`, real perturbations) → L (`RUN_SMOKE=False`, saved real analysis).

In [ ]:
RUN_SMOKE = True
RUN_FULL = True
RESUME = True
OVERWRITE = False
from pathlib import Path
import sys, subprocess, json
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT=Path.cwd().resolve()
while not (ROOT/'src/week12').is_dir():
    if ROOT==ROOT.parent: raise FileNotFoundError('Open within project')
    ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.week12.protocol import load, guard
from src.week12_attribution.protocol import directory
cfg=load();guard()
if OVERWRITE or not RESUME: raise ValueError('RESUME=True, OVERWRITE=False required')
SCRIPT=ROOT/'scripts/week12/run_attribution.py'
def run(stage, *args):
    subprocess.run([str(ROOT/'sara_env/bin/python'),str(SCRIPT),'--stage',stage,*args],cwd=ROOT,check=True)


In [ ]:
# Setup does not load an LM or generate answers.
if RUN_SMOKE:
    run('setup','--smoke')
else:
    run('setup')
p=ROOT/'results/week12/attribution/ATTRIBUTION_QUERY_FREEZE.json'
if p.exists():
    frozen=json.loads(p.read_text())
    display(pd.DataFrame([dict(r,dataset=d) for d,rs in frozen['selected'].items() for r in rs]))
    display(pd.DataFrame([dict(dataset=x['dataset'],budget=x['budget'],comparator=x['chosen']) for x in frozen['comparator_results']]))
    print('Confirmation overlap = 0; membership validated before freezing.')
    plan=directory(cfg)/'cost_plan.json'
    if plan.exists(): display(pd.DataFrame([json.loads(plan.read_text())]))
else:
    print('Real query freeze pending; set RUN_SMOKE=False and Run All (cached setup only).')
